In [10]:
pip install tensorflow keras matplotlib scikit-learn

  Using cached numpy-2.0.2-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached numpy-1.26.4-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
INFO: pip is looking at multiple versions of scipy to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.5 MB/s eta 0:00:00 MB/s eta 0:00:0101
Using cached numpy-1.26.4-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 5.8 MB/s eta 0:00:00m eta 0:00:010:00:01m
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.5
    Uninstalling numpy-1.23.5:
      Successfully uninstalled numpy-1.23.5
  Attempting uninstall: scipy
    Found existing installation: scipy 1.9.3
    Uninstalling scipy-1.9.3:
      Successfully uninstalled 

In [18]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Image size and batch size
img_size = (128, 128)
batch_size = 32

# Data generators with train/validation split
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

dataset_path = "../Downloads/garbage_classification"   # adjust this to your actual folder path

train_gen = train_datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    subset="training"
)

val_gen = train_datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    subset="validation"
)


Found 12415 images belonging to 12 classes.
Found 3100 images belonging to 12 classes.


In [24]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout

# Load MobileNetV2 without the top classifier
base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(128,128,3))

# Freeze base model layers (so we don’t retrain them initially)
for layer in base_model.layers:
    layer.trainable = False


In [27]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)
preds = Dense(train_gen.num_classes, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=preds)


In [29]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [31]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    verbose=1
)

Epoch 1/10


/home/admin1/anaconda3/lib/python3.9/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


388/388 ━━━━━━━━━━━━━━━━━━━━ 64s 160ms/step - accuracy: 0.7177 - loss: 0.9604 - val_accuracy: 0.8674 - val_loss: 0.4114
Epoch 2/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 63s 161ms/step - accuracy: 0.8993 - loss: 0.3271 - val_accuracy: 0.8929 - val_loss: 0.3183
Epoch 3/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 63s 163ms/step - accuracy: 0.9220 - loss: 0.2527 - val_accuracy: 0.8945 - val_loss: 0.3281
Epoch 4/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 64s 164ms/step - accuracy: 0.9361 - loss: 0.1991 - val_accuracy: 0.8997 - val_loss: 0.3109
Epoch 5/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 82s 165ms/step - accuracy: 0.9439 - loss: 0.1720 - val_accuracy: 0.9052 - val_loss: 0.3091
Epoch 6/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 65s 168ms/step - accuracy: 0.9477 - loss: 0.1496 - val_accuracy: 0.8958 - val_loss: 0.3314
Epoch 7/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 66s 169ms/step - accuracy: 0.9527 - loss: 0.1358 - val_accuracy: 0.9052 - val_loss: 0.3142
Epoch 8/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 67s 172ms/step - accuracy: 0.9622 - loss: 0.1169 - val

In [33]:
model.save("mobilenetv2_garbage_classifier.h5")

In [35]:
# Save in new Keras format
model.save("mobilenetv2_garbage_classifier.keras")

In [37]:
for layer in base_model.layers[-20:]:  # unfreeze last 20 layers
    layer.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), 
              loss="categorical_crossentropy", 
              metrics=["accuracy"])

history_finetune = model.fit(train_gen, validation_data=val_gen, epochs=5)


Epoch 1/5
388/388 ━━━━━━━━━━━━━━━━━━━━ 77s 190ms/step - accuracy: 0.8548 - loss: 0.5621 - val_accuracy: 0.9019 - val_loss: 0.3403
Epoch 2/5
388/388 ━━━━━━━━━━━━━━━━━━━━ 72s 186ms/step - accuracy: 0.9120 - loss: 0.3042 - val_accuracy: 0.9032 - val_loss: 0.3286
Epoch 3/5
388/388 ━━━━━━━━━━━━━━━━━━━━ 74s 191ms/step - accuracy: 0.9272 - loss: 0.2244 - val_accuracy: 0.9045 - val_loss: 0.3283
Epoch 4/5
388/388 ━━━━━━━━━━━━━━━━━━━━ 73s 187ms/step - accuracy: 0.9378 - loss: 0.1868 - val_accuracy: 0.9084 - val_loss: 0.3162
Epoch 5/5
388/388 ━━━━━━━━━━━━━━━━━━━━ 72s 186ms/step - accuracy: 0.9461 - loss: 0.1737 - val_accuracy: 0.9106 - val_loss: 0.3185


In [45]:
from tensorflow.keras.models import load_model

# Load the model
model = load_model("mobilenetv2_garbage_classifier.keras")

# Re-compile with your settings
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

# Now you can evaluate or train
val_loss, val_acc = model.evaluate(val_gen)
print("Validation Accuracy:", val_acc)


/home/admin1/anaconda3/lib/python3.9/site-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 6 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


97/97 ━━━━━━━━━━━━━━━━━━━━ 13s 127ms/step - accuracy: 0.9082 - loss: 0.3290
Validation Accuracy: 0.906129002571106


In [39]:
from tensorflow.keras.models import load_model

model = load_model("mobilenetv2_garbage_classifier.h5")


In [41]:
import numpy as np
from tensorflow.keras.preprocessing import image

img_path = "../Downloads/plastic_bottle.jpeg"   # replace with your image path
img = image.load_img(img_path, target_size=(128,128))   # resize to match training
img_array = image.img_to_array(img) / 255.0             # normalize
img_array = np.expand_dims(img_array, axis=0)           # add batch dimension


In [43]:
prediction = model.predict(img_array)
predicted_class = np.argmax(prediction)

# Get class labels from your generator
class_labels = list(train_gen.class_indices.keys())

print("Predicted:", class_labels[predicted_class])


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 542ms/step
Predicted: plastic
